# Day 23 — Software Performance / Final Validation
This notebook evaluates the frozen Day 22 pipeline on the reserved chronological test set and compares it with a constant-mean baseline. No tuning or model changes are performed.

In [ ]:
import time, os, joblib, numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

In [ ]:
df=pd.read_csv('water_potability_temporal_synthetic.csv')
df['timestamp']=pd.to_datetime(df['timestamp'],errors='coerce')
data=df.dropna(subset=['timestamp']).sort_values('timestamp',kind='stable').reset_index(drop=True)
data['target_next_ph']=data['ph'].shift(-1)
data=data.dropna(subset=['target_next_ph']).reset_index(drop=True)
n=len(data); a=int(n*.70); b=int(n*.85)
train,val,test=data.iloc[:a],data.iloc[a:b],data.iloc[b:]
print('Train / validation / test:',len(train),len(val),len(test))

In [ ]:
model=joblib.load('Day22_Final_Linear_Regression_Pipeline.joblib')
dev=pd.concat([train,val],ignore_index=True)
baseline=DummyRegressor(strategy='mean').fit(dev[['ph']],dev.target_next_ph)
X=test[['ph']]; y=test.target_next_ph.to_numpy()
pred=model.predict(X); base_pred=baseline.predict(X)
def report(y,p): return {'MAE':mean_absolute_error(y,p),'RMSE':np.sqrt(mean_squared_error(y,p))}
print('Frozen model:',report(y,pred))
print('Constant-mean baseline:',report(y,base_pred))

In [ ]:
for name,est in [('Linear Regression',model),('Constant mean',baseline)]:
    est.predict(X.iloc[:10])
    times=[]
    for _ in range(200):
        t=time.perf_counter_ns(); est.predict(X); times.append((time.perf_counter_ns()-t)/1e6)
    print(name,'median batch ms:',np.median(times),'p95 batch ms:',np.percentile(times,95))
print('Frozen pipeline file size (KB):',os.path.getsize('Day22_Final_Linear_Regression_Pipeline.joblib')/1024)

In [ ]:
results=pd.DataFrame({'actual_next_ph':y,'linear_regression_prediction':pred,'constant_mean_prediction':base_pred})
results['linear_regression_absolute_error']=abs(results.actual_next_ph-results.linear_regression_prediction)
results['constant_mean_absolute_error']=abs(results.actual_next_ph-results.constant_mean_prediction)
results.head()

**Interpretation:** compare the test metrics and runtime. Do not claim real underwater forecasting: the timestamp column is synthetic and the source dataset is static.